# 야간 운전 시각 개선 시스템 — 데이터 준비 (Colab)

LOL Dataset 다운로드 + 커스텀 데이터 통일 구조 변환을 수행합니다.

## 실행 순서
1. 환경 설치
2. Google Drive 마운트
3. 저장소 코드 로드
4. LOL Dataset 다운로드
5. 커스텀 데이터 복사
6. 통일 구조로 변환 (`data/processed/`)
7. 데이터셋 샘플 확인

> 학습은 **train_colab.ipynb** 에서 진행합니다.

---
## 1. 환경 설치

In [ ]:
# GPU 확인
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# 필요 패키지 설치
%pip install -q gdown scikit-image tqdm

---
## 2. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Drive 내 경로 설정 (본인 경로에 맞게 수정)
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/night-vision'
DRIVE_CUSTOM_DATA  = '/content/drive/MyDrive/night-vision-data'  # 커스텀 데이터 위치

import os
print('Drive 마운트 완료.')

---
## 3. 저장소 코드 로드

**방법 A** (권장): GitHub에서 클론  
**방법 B**: Drive에 올린 코드를 Colab으로 복사

In [ ]:
import os

REPO_URL    = 'https://github.com/mia2583/night-vision.git'
PROJECT_DIR = '/content/night-vision'

# ── 방법 A: GitHub 클론 ──────────────────────────
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

# ── 방법 B: Drive에서 복사 (A 실패 시 주석 해제) ──
# import shutil
# shutil.copytree(DRIVE_PROJECT_ROOT, PROJECT_DIR, dirs_exist_ok=True)

os.chdir(PROJECT_DIR)
print('작업 디렉터리:', os.getcwd())

---
## 4. LOL Dataset 다운로드

In [ ]:
import os

LOL_DIR = 'data/lol'
os.makedirs(LOL_DIR, exist_ok=True)

from utils.download_lol import download_lol_dataset

try:
    download_lol_dataset(LOL_DIR)
except RuntimeError as e:
    print(f'자동 다운로드 실패: {e}')
    print('LOL 없이 커스텀 데이터만으로 진행합니다.')

---
## 5. 커스텀 데이터 복사

커스텀 데이터 구조:
```
DRIVE_CUSTOM_DATA/
├── train_input_img/   (빛 번짐 있는 원본)
├── train_label_img/   (개선된 참고 이미지)
└── test_input_img/    (테스트 이미지)
```

In [ ]:
import os
import shutil

CUSTOM_DIR = 'data/custom'
os.makedirs(CUSTOM_DIR, exist_ok=True)

if os.path.exists(DRIVE_CUSTOM_DATA):
    shutil.copytree(DRIVE_CUSTOM_DATA, CUSTOM_DIR, dirs_exist_ok=True)
    for folder in ['train_input_img', 'train_label_img', 'test_input_img']:
        path = os.path.join(CUSTOM_DIR, folder)
        if os.path.exists(path):
            count = len([f for f in os.listdir(path) if f.endswith('.png')])
            print(f'  {folder}: {count}장')
else:
    print(f'커스텀 데이터 없음: {DRIVE_CUSTOM_DATA}')
    print('LOL 데이터만으로 진행합니다.')

---
## 6. 통일 구조로 변환

In [ ]:
from utils.prepare_data import prepare_data

DATA_DIR = 'data/processed'

meta = prepare_data(
    lol_dir=LOL_DIR,
    custom_dir=CUSTOM_DIR,
    output_dir=DATA_DIR,
    force=False,
)

print('\n데이터 준비 완료!')
print(f"  학습: {meta['total_train']}쌍")
print(f"  검증: {meta['total_val']}쌍")
print(f"  테스트: {meta['total_test']}장")

---
## 7. 데이터셋 샘플 확인

In [ ]:
import matplotlib.pyplot as plt
from utils.data_loader import NightVisionDataset
from utils.augmentation import get_transform

train_ds = NightVisionDataset(DATA_DIR, split='train', transform=get_transform('train'))
val_ds   = NightVisionDataset(DATA_DIR, split='val',   transform=get_transform('val'))
test_ds  = NightVisionDataset(DATA_DIR, split='test',  transform=get_transform('test'))

print(f'train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}')

# 샘플 시각화
sample  = train_ds[0]
inp_np  = sample['input'].permute(1, 2, 0).numpy()
tgt_np  = sample['target'].permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(inp_np); axes[0].set_title('Input (저조도/빛 번짐)')
axes[1].imshow(tgt_np); axes[1].set_title('Target (개선된 이미지)')
for ax in axes: ax.axis('off')
plt.suptitle(f"샘플: {sample['filename']}")
plt.tight_layout()
plt.show()

print('\n다음 단계: train_colab.ipynb 에서 모델 학습을 진행하세요.')